# MSTR Enterprise Value and NAV Calculation

This notebook calculates:
1. Enterprise Value (EV) by summing all market caps (shares × price) from strategy.com
2. Current Bitcoin price from Yahoo Finance
3. Net Asset Value (NAV) including cash position
4. Stock price and premium percentage
5. Two NAV scenarios: with and without convertible debt conversion

In [1]:
# ============================================================================
# DATA FETCHING - MicroStrategy Financial Data from strategy.com
# ============================================================================

import requests
from bs4 import BeautifulSoup
import json
import yfinance as yf

def fetch_microstrategy_data():
    """
    Fetch MicroStrategy financial data directly from strategy.com
    """
    print("Fetching MicroStrategy financial data from https://www.strategy.com/...")
    data = {}
    
    try:
        url = "https://www.strategy.com/"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
        }
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Find the __NEXT_DATA__ script tag containing all the data
            script_tag = soup.find('script', {'id': '__NEXT_DATA__'})
            
            if script_tag:
                next_data = json.loads(script_tag.string)
                
                # Extract btcTrackerData from the JSON structure
                btc_tracker_data = next_data.get('props', {}).get('pageProps', {}).get('btcTrackerData', [])
                
                if btc_tracker_data and len(btc_tracker_data) > 0:
                    latest_data = btc_tracker_data[0]  # Get the latest entry
                    
                    # Extract Bitcoin holdings
                    if 'btc_holdings' in latest_data:
                        data['bitcoin_holdings'] = int(latest_data['btc_holdings'])
                        print(f"  ✓ Bitcoin holdings: {data['bitcoin_holdings']:,} BTC")
                    
                    # Extract cash reserve
                    if 'cash' in latest_data:
                        data['cash'] = float(latest_data['cash'])
                        print(f"  ✓ Cash reserve: ${data['cash']:,.0f}")
                    
                    # Extract MSTR shares outstanding
                    # Try different possible keys
                    if 'mstr_shares' in latest_data:
                        data['mstr_shares'] = int(latest_data['mstr_shares'])
                        print(f"  ✓ MSTR shares outstanding: {data['mstr_shares']:,}")
                    elif 'shares_outstanding' in latest_data:
                        data['mstr_shares'] = int(latest_data['shares_outstanding'])
                        print(f"  ✓ MSTR shares outstanding: {data['mstr_shares']:,}")
                    
                    # Extract preferred stock data (STRC, STRD, STRE, STRK, STRF)
                    preferred_series = ['strc', 'strd', 'stre', 'strk', 'strf']
                    
                    for series in preferred_series:
                        metrics_key = f'{series}_metrics'
                        
                        if metrics_key in latest_data:
                            series_data = latest_data[metrics_key]
                            
                            # Extract shares
                            if 'shares' in series_data:
                                shares = int(series_data['shares'])
                                data[f'{series}_shares'] = shares
                                print(f"  ✓ {series.upper()} shares: {shares:,}")
                    
                    print(f"  ✓ Data date: {latest_data.get('as_of_date', 'N/A')}")
                else:
                    print("  ✗ No btcTrackerData found in response")
            else:
                print("  ✗ Could not find __NEXT_DATA__ script tag")
        else:
            print(f"  ✗ HTTP {response.status_code}: Could not fetch from strategy.com")
            
        # Fetch convertible debt data from the debt page
        print("\nFetching convertible debt data from https://www.strategy.com/debt...")
        try:
            debt_url = "https://www.strategy.com/debt"
            debt_response = requests.get(debt_url, headers=headers, timeout=15)
            
            if debt_response.status_code == 200:
                debt_soup = BeautifulSoup(debt_response.text, 'html.parser')
                debt_script_tag = debt_soup.find('script', {'id': '__NEXT_DATA__'})
                
                if debt_script_tag:
                    debt_next_data = json.loads(debt_script_tag.string)
                    debt_page_props = debt_next_data.get('props', {}).get('pageProps', {})
                    
                    # Look for debt data in various possible locations
                    data['convertible_debt'] = []
                    
                    # Check for debt data - it's in 'convertData' key
                    if 'convertData' in debt_page_props:
                        data['convertible_debt'] = debt_page_props['convertData']
                        print(f"  ✓ Found convertible debt data: {len(data['convertible_debt'])} issues")
                    else:
                        # Fallback: check for debt array/list in pageProps
                        for key in debt_page_props.keys():
                            if 'debt' in key.lower() or 'bond' in key.lower() or 'convert' in key.lower():
                                debt_data = debt_page_props[key]
                                if isinstance(debt_data, list):
                                    data['convertible_debt'] = debt_data
                                    print(f"  ✓ Found convertible debt data: {len(debt_data)} issues")
                                    break
                        
                        # If still not found, check the entire pageProps structure
                        if not data['convertible_debt']:
                            # Look for arrays that might contain debt data
                            for key, value in debt_page_props.items():
                                if isinstance(value, list) and len(value) > 0:
                                    # Check if items in the list look like debt objects
                                    if isinstance(value[0], dict):
                                        # Check if it has debt-like fields
                                        first_item = value[0]
                                        if any(field in first_item for field in ['principal', 'face_value', 'amount', 'notional', 'strike_price', 'strike']):
                                            data['convertible_debt'] = value
                                            print(f"  ✓ Found convertible debt data: {len(value)} issues")
                                            break
                    
                    if data['convertible_debt']:
                        print(f"  ✓ Total convertible debt issues: {len(data['convertible_debt'])}")
                        for i, debt in enumerate(data['convertible_debt'], 1):
                            # Helper function to get value from multiple possible keys, handling nested dicts and strings
                            def get_value(d, *keys):
                                for key in keys:
                                    if key in d and d[key] is not None:
                                        val = d[key]
                                        # Try to convert string to float if needed
                                        if isinstance(val, str):
                                            try:
                                                # Remove $, commas, etc.
                                                val = val.replace('$', '').replace(',', '').strip()
                                                return float(val)
                                            except:
                                                return val
                                        return val
                                return None
                            
                            # Helper to search nested dicts
                            def search_nested(d, target_keys):
                                if isinstance(d, dict):
                                    for key, value in d.items():
                                        if any(tk.lower() in key.lower() for tk in target_keys):
                                            if isinstance(value, (int, float)):
                                                return value
                                            elif isinstance(value, str):
                                                try:
                                                    return float(value.replace('$', '').replace(',', '').strip())
                                                except:
                                                    pass
                                        if isinstance(value, dict):
                                            result = search_nested(value, target_keys)
                                            if result is not None:
                                                return result
                                return None
                            
                            # Extract principal - it's 'notional' in the actual data
                            principal = get_value(debt, 'notional', 'principal', 'face_value', 'faceValue', 'amount', 
                                                 'par_value', 'parValue', 'total_principal', 'totalPrincipal')
                            if principal is None:
                                principal = search_nested(debt, ['notional', 'principal', 'face', 'amount', 'par'])
                            principal = principal or 0
                            
                            # Extract conversion price - it's 'strike_price' in the actual data
                            conversion_price = get_value(debt, 'strike_price', 'strikePrice', 'strike', 
                                                       'conversion_price', 'conversionPrice', 'conversion_strike', 'conversionStrike',
                                                       'conversion_rate', 'conversionRate', 'conversion', 
                                                       'conversionPricePerShare', 'conversion_price_per_share',
                                                       'share_conversion_price', 'shareConversionPrice', 'price')
                            if conversion_price is None:
                                conversion_price = search_nested(debt, ['strike', 'conversion', 'price'])
                            conversion_price = conversion_price or 0
                            
                            # Extract maturity - it's 'maturity_date' in the actual data
                            maturity = get_value(debt, 'maturity_date', 'maturityDate', 'maturity') or 'N/A'
                            
                            # Extract coupon - it's 'coupon' in the actual data
                            coupon = get_value(debt, 'coupon', 'coupon_rate', 'couponRate', 'interest_rate', 
                                             'interestRate') or 0
                            
                            # Calculate conversion ratio from principal and conversion price if needed
                            conversion_ratio = 0
                            if principal > 0 and conversion_price > 0:
                                # Conversion ratio = principal / conversion_price (shares per bond)
                                conversion_ratio = principal / conversion_price
                            
                            print(f"    Issue {i}: Principal=${principal:,.0f}, Conversion Price=${conversion_price:,.2f}, Coupon={coupon}%, Maturity={maturity}")
                    else:
                        print("  ⚠ No convertible debt data found in debt page")
                        # Debug: print available keys
                        print("  Available keys in debt pageProps:")
                        for key in sorted(debt_page_props.keys())[:20]:  # Show first 20 keys
                            print(f"    - {key}: {type(debt_page_props[key])}")
                else:
                    print("  ✗ Could not find __NEXT_DATA__ script tag in debt page")
            else:
                print(f"  ✗ HTTP {debt_response.status_code}: Could not fetch from strategy.com/debt")
        except Exception as e:
            print(f"  ⚠ Error fetching debt data: {e}")
            import traceback
            traceback.print_exc()
        
        # Calculate total convertible debt principal
        if data.get('convertible_debt'):
            total_debt_principal = 0
            for debt in data['convertible_debt']:
                notional = debt.get('notional', 0) or 0
                total_debt_principal += notional
            data['total_convertible_debt_principal'] = total_debt_principal
            print(f"\n  ✓ Total convertible debt principal: ${total_debt_principal:,.0f}")
        else:
            data['total_convertible_debt_principal'] = 0
        
        # Fetch shares conversion data from the shares page
        print("\nFetching share conversion data from https://www.strategy.com/shares...")
        try:
            shares_url = "https://www.strategy.com/shares"
            shares_response = requests.get(shares_url, headers=headers, timeout=15)
            
            if shares_response.status_code == 200:
                shares_soup = BeautifulSoup(shares_response.text, 'html.parser')
                shares_script_tag = shares_soup.find('script', {'id': '__NEXT_DATA__'})
                
                if shares_script_tag:
                    shares_next_data = json.loads(shares_script_tag.string)
                    shares_page_props = shares_next_data.get('props', {}).get('pageProps', {})
                    
                    # Get shares data
                    shares_data = shares_page_props.get('shares', [])
                    
                    if shares_data:
                        # Find the most recent entry by date (not just first with conversion data)
                        from datetime import datetime
                        most_recent_entry = None
                        most_recent_date = None
                        
                        for entry in shares_data:
                            date_str = entry.get('date', '')
                            if date_str:
                                try:
                                    entry_date = datetime.strptime(date_str, '%Y-%m-%d')
                                    if most_recent_date is None or entry_date > most_recent_date:
                                        most_recent_date = entry_date
                                        most_recent_entry = entry
                                except:
                                    pass
                        
                        if most_recent_entry:
                            print(f"  ✓ Using most recent shares data: {most_recent_entry.get('title', 'N/A')} ({most_recent_entry.get('date', 'N/A')})")
                            
                            # Get shares outstanding components (all in thousands, multiply by 1000)
                            basic_shares = most_recent_entry.get('basic_shares_outstanding', 0) or 0
                            options_outstanding = most_recent_entry.get('options_outstanding', 0) or 0
                            rsu_psu_unvested = most_recent_entry.get('rsu_psu_unvested', 0) or 0
                            
                            # Calculate fully diluted shares = basic + options + rsu/psu
                            if basic_shares > 0:
                                basic_shares_actual = int(basic_shares * 1000)
                                options_actual = int(options_outstanding * 1000) if options_outstanding > 0 else 0
                                rsu_psu_actual = int(rsu_psu_unvested * 1000) if rsu_psu_unvested > 0 else 0
                                
                                # Fully diluted shares outstanding
                                data['mstr_shares'] = basic_shares_actual + options_actual + rsu_psu_actual
                                
                                print(f"  ✓ MSTR basic shares outstanding: {basic_shares_actual:,}")
                                if options_actual > 0:
                                    print(f"  ✓ Options outstanding: {options_actual:,}")
                                if rsu_psu_actual > 0:
                                    print(f"  ✓ RSU/PSU unvested: {rsu_psu_actual:,}")
                                print(f"  ✓ Total fully diluted shares outstanding: {data['mstr_shares']:,}")
                            
                            # Get conversion shares
                            conversion_shares = {
                                '2028': most_recent_entry.get('converts_shares_2028'),
                                '2029': most_recent_entry.get('converts_shares_2029'),
                                '2030': most_recent_entry.get('converts_shares_2030'),
                                '2030_b': most_recent_entry.get('converts_shares_2030_b'),
                                '2031': most_recent_entry.get('converts_shares_2031'),
                                '2032': most_recent_entry.get('converts_shares_2032')
                            }
                        else:
                            conversion_shares = {}
                        
                        # Map conversion shares to debt issues
                        if conversion_shares and data.get('convertible_debt'):
                            for debt in data['convertible_debt']:
                                maturity_date = debt.get('maturity_date', '')
                                title = debt.get('title', '').lower()
                                
                                # Extract year from maturity date
                                if maturity_date:
                                    year = maturity_date.split('-')[0] if '-' in maturity_date else maturity_date[:4]
                                    
                                    # Match by year and title
                                    # Note: shares are stored in thousands, so multiply by 1000
                                    if year == '2028':
                                        shares_val = conversion_shares.get('2028', 0) or 0
                                        debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                    elif year == '2029':
                                        shares_val = conversion_shares.get('2029', 0) or 0
                                        debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                    elif year == '2030':
                                        if 'b' in title or '2030 b' in title:
                                            shares_val = conversion_shares.get('2030_b', 0) or 0
                                            debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                        else:
                                            shares_val = conversion_shares.get('2030', 0) or 0
                                            debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                    elif year == '2031':
                                        shares_val = conversion_shares.get('2031', 0) or 0
                                        debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                    elif year == '2032':
                                        shares_val = conversion_shares.get('2032', 0) or 0
                                        debt['shares_on_conversion'] = shares_val * 1000 if shares_val > 0 else 0
                                    else:
                                        debt['shares_on_conversion'] = 0
                                else:
                                    debt['shares_on_conversion'] = 0
                    else:
                        print("  ⚠ No shares data found")
                else:
                    print("  ✗ Could not find __NEXT_DATA__ script tag in shares page")
            else:
                print(f"  ✗ HTTP {shares_response.status_code}: Could not fetch from strategy.com/shares")
        except Exception as e:
            print(f"  ⚠ Error fetching shares data: {e}")
            import traceback
            traceback.print_exc()
            
    except Exception as e:
        print(f"  ✗ Error fetching from strategy.com: {e}")
        import traceback
        traceback.print_exc()
    
    return data

# Fetch data
mstr_data = fetch_microstrategy_data()


Fetching MicroStrategy financial data from https://www.strategy.com/...
  ✓ Bitcoin holdings: 818,869 BTC
  ✓ Cash reserve: $2,250,000,000
  ✓ STRC shares: 85,374,904
  ✓ STRD shares: 14,024,221
  ✓ STRE shares: 9,127,175
  ✓ STRK shares: 14,020,744
  ✓ STRF shares: 12,839,689
  ✓ Data date: 2026-05-11

Fetching convertible debt data from https://www.strategy.com/debt...
  ✓ Found convertible debt data: 6 issues
  ✓ Total convertible debt issues: 6
    Issue 1: Principal=$1,010,000,000, Conversion Price=$183.19, Coupon=0.625%, Maturity=2028-09-16
    Issue 2: Principal=$3,000,000,000, Conversion Price=$672.40, Coupon=0%, Maturity=2029-12-02
    Issue 3: Principal=$2,000,000,000, Conversion Price=$433.43, Coupon=0%, Maturity=2030-03-02
    Issue 4: Principal=$800,000,000, Conversion Price=$149.77, Coupon=0.625%, Maturity=2030-03-16
    Issue 5: Principal=$603,750,000, Conversion Price=$232.72, Coupon=0.875%, Maturity=2031-03-16
    Issue 6: Principal=$800,000,000, Conversion Price=$204.

In [2]:
# ============================================================================
# FETCH CURRENT PRICES FROM YAHOO FINANCE
# ============================================================================

print("\nFetching current prices from Yahoo Finance...")

# Fetch Bitcoin price
try:
    btc_ticker = yf.Ticker("BTC-USD")
    btc_data = btc_ticker.history(period="1d")
    if len(btc_data) > 0:
        CURRENT_BITCOIN_PRICE = float(btc_data['Close'].iloc[-1])
        print(f"  ✓ Current Bitcoin price: ${CURRENT_BITCOIN_PRICE:,.2f}")
    else:
        CURRENT_BITCOIN_PRICE = 92000  # Fallback
        print(f"  ⚠ Using fallback Bitcoin price: ${CURRENT_BITCOIN_PRICE:,.2f}")
except Exception as e:
    CURRENT_BITCOIN_PRICE = 92000  # Fallback
    print(f"  ⚠ Error fetching Bitcoin price: {e}")
    print(f"  Using fallback Bitcoin price: ${CURRENT_BITCOIN_PRICE:,.2f}")

# Fetch EUR/USD exchange rate for STRE (denominated in EUR)
print("\nFetching EUR/USD exchange rate from Yahoo Finance...")
try:
    eur_usd_ticker = yf.Ticker("EURUSD=X")
    eur_usd_data = eur_usd_ticker.history(period="1d")
    if len(eur_usd_data) > 0:
        EUR_USD_RATE = float(eur_usd_data['Close'].iloc[-1])
        print(f"  ✓ EUR/USD exchange rate: {EUR_USD_RATE:.4f}")
    else:
        EUR_USD_RATE = 1.08  # Fallback
        print(f"  ⚠ Using fallback EUR/USD rate: {EUR_USD_RATE:.4f}")
except Exception as e:
    EUR_USD_RATE = 1.08  # Fallback
    print(f"  ⚠ Error fetching EUR/USD rate: {e}")
    print(f"  Using fallback EUR/USD rate: {EUR_USD_RATE:.4f}")

# Fetch MSTR price
print("\nFetching MSTR stock price from Yahoo Finance...")
try:
    mstr_ticker = yf.Ticker("MSTR")
    mstr_info = mstr_ticker.info
    mstr_price_data = mstr_ticker.history(period="1d")
    
    if len(mstr_price_data) > 0:
        mstr_data['mstr_price'] = float(mstr_price_data['Close'].iloc[-1])
        print(f"  ✓ MSTR price: ${mstr_data['mstr_price']:,.2f}")
    
    # Also get shares outstanding from Yahoo if not available
    if 'mstr_shares' not in mstr_data or mstr_data.get('mstr_shares', 0) == 0:
        if 'sharesOutstanding' in mstr_info:
            mstr_data['mstr_shares'] = int(mstr_info['sharesOutstanding'])
            print(f"  ✓ MSTR shares outstanding (from Yahoo): {mstr_data['mstr_shares']:,}")
except Exception as e:
    print(f"  ⚠ Error fetching MSTR data: {e}")

# Fetch preferred stock prices
preferred_series = ['strc', 'strd', 'stre', 'strk', 'strf']
for series in preferred_series:
    price_key = f'{series}_price'
    if price_key not in mstr_data or mstr_data.get(price_key, 0) == 0:
        print(f"\nFetching {series.upper()} price from Yahoo Finance...")
        try:
            ticker = yf.Ticker(series.upper())
            price_data = ticker.history(period="1d")
            if len(price_data) > 0:
                mstr_data[price_key] = float(price_data['Close'].iloc[-1])
                print(f"  ✓ {series.upper()} price: ${mstr_data[price_key]:,.2f}")
            else:
                print(f"  ⚠ Could not fetch {series.upper()} price from Yahoo Finance")
        except Exception as e:
            print(f"  ⚠ Error fetching {series.upper()} price: {e}")

# Default STRE to par value of 100 EUR per share if price not available
if 'stre_price' not in mstr_data or mstr_data.get('stre_price', 0) == 0:
    STRE_PAR_VALUE_EUR = 100
    mstr_data['stre_price'] = STRE_PAR_VALUE_EUR * EUR_USD_RATE
    print(f"\n  ⚠ STRE price not available, defaulting to par value:")
    print(f"  ✓ STRE par value: €{STRE_PAR_VALUE_EUR:.2f}")
    print(f"  ✓ STRE price (converted to USD): ${mstr_data['stre_price']:,.2f} (€{STRE_PAR_VALUE_EUR:.2f} × {EUR_USD_RATE:.4f})")



Fetching current prices from Yahoo Finance...
  ✓ Current Bitcoin price: $79,339.44

Fetching EUR/USD exchange rate from Yahoo Finance...
  ✓ EUR/USD exchange rate: 1.1721

Fetching MSTR stock price from Yahoo Finance...
  ✓ MSTR price: $178.03

Fetching STRC price from Yahoo Finance...
  ✓ STRC price: $100.00

Fetching STRD price from Yahoo Finance...
  ✓ STRD price: $76.99

Fetching STRE price from Yahoo Finance...


$STRE: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


  ⚠ Could not fetch STRE price from Yahoo Finance

Fetching STRK price from Yahoo Finance...
  ✓ STRK price: $76.88

Fetching STRF price from Yahoo Finance...
  ✓ STRF price: $100.36

  ⚠ STRE price not available, defaulting to par value:
  ✓ STRE par value: €100.00
  ✓ STRE price (converted to USD): $117.21 (€100.00 × 1.1721)


In [3]:
# ============================================================================
# CALCULATE MARKET CAPS AND ENTERPRISE VALUE (EV)
# ============================================================================

# Get MSTR shares and price
MSTR_SHARES = mstr_data.get('mstr_shares', 0)
MSTR_PRICE = mstr_data.get('mstr_price', 0)

# Calculate MSTR market cap: shares * price
MSTR_MARKET_CAP = MSTR_SHARES * MSTR_PRICE if (MSTR_SHARES > 0 and MSTR_PRICE > 0) else 0

# Calculate preferred stock market caps: shares * price for each series
preferred_series = ['strc', 'strd', 'stre', 'strk', 'strf']
TOTAL_PREFERRED_MARKET_CAP = 0

print("=" * 70)
print("MARKET CAP CALCULATIONS")
print("=" * 70)
print(f"MSTR Common Stock:")
print(f"  Shares: {MSTR_SHARES:,}")
print(f"  Price: ${MSTR_PRICE:,.2f}")
if MSTR_SHARES > 0 and MSTR_PRICE > 0:
    print(f"  Market Cap = {MSTR_SHARES:,} × ${MSTR_PRICE:,.2f} = ${MSTR_MARKET_CAP:,.0f}")
else:
    print(f"  ⚠ Cannot calculate market cap (missing shares or price)")

print(f"\nPreferred Stock:")
for series in preferred_series:
    shares_key = f'{series}_shares'
    price_key = f'{series}_price'
    
    shares = mstr_data.get(shares_key, 0)
    price = mstr_data.get(price_key, 0)
    
    if shares > 0:
        if price > 0:
            mcap = shares * price
            TOTAL_PREFERRED_MARKET_CAP += mcap
            print(f"  {series.upper()}: {shares:,} shares × ${price:,.2f} = ${mcap:,.0f}")
        else:
            print(f"  {series.upper()}: {shares:,} shares (price not available)")

# Get debt and cash for EV calculation
TOTAL_DEBT = mstr_data.get('total_convertible_debt_principal', 0)
CASH = mstr_data.get('cash', 0)

# Calculate Enterprise Value = Market Cap + Debt - Cash
# EV = Equity Market Cap + Preferred Market Cap + Debt - Cash
EQUITY_MARKET_CAP = MSTR_MARKET_CAP + TOTAL_PREFERRED_MARKET_CAP
ENTERPRISE_VALUE = EQUITY_MARKET_CAP + TOTAL_DEBT - CASH

print(f"\n" + "=" * 70)
print("ENTERPRISE VALUE CALCULATION")
print("=" * 70)
print(f"MSTR Common Stock Market Cap: ${MSTR_MARKET_CAP:,.0f}")
print(f"Total Preferred Stock Market Cap: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"Total Equity Market Cap: ${EQUITY_MARKET_CAP:,.0f}")
print(f"Total Convertible Debt: ${TOTAL_DEBT:,.0f}")
print(f"Cash Reserve: ${CASH:,.0f}")
print(f"\nEnterprise Value (EV) = Equity Market Cap + Debt - Cash")
print(f"Enterprise Value (EV) = ${EQUITY_MARKET_CAP:,.0f} + ${TOTAL_DEBT:,.0f} - ${CASH:,.0f}")
print(f"Enterprise Value (EV): ${ENTERPRISE_VALUE:,.0f}")
print("=" * 70)


MARKET CAP CALCULATIONS
MSTR Common Stock:
  Shares: 355,898,000
  Price: $178.03
  Market Cap = 355,898,000 × $178.03 = $63,360,520,506

Preferred Stock:
  STRC: 85,374,904 shares × $100.00 = $8,537,490,400
  STRD: 14,024,221 shares × $76.99 = $1,079,724,745
  STRE: 9,127,175 shares × $117.21 = $1,069,757,944
  STRK: 14,020,744 shares × $76.88 = $1,077,914,760
  STRF: 12,839,689 shares × $100.36 = $1,288,591,196

ENTERPRISE VALUE CALCULATION
MSTR Common Stock Market Cap: $63,360,520,506
Total Preferred Stock Market Cap: $13,053,479,045
Total Equity Market Cap: $76,413,999,550
Total Convertible Debt: $8,213,750,000
Cash Reserve: $2,250,000,000

Enterprise Value (EV) = Equity Market Cap + Debt - Cash
Enterprise Value (EV) = $76,413,999,550 + $8,213,750,000 - $2,250,000,000
Enterprise Value (EV): $82,377,749,550


In [4]:
# ============================================================================
# ANALYZE CONVERTIBLE DEBT
# ============================================================================

BITCOIN_HOLDINGS = mstr_data.get('bitcoin_holdings', 0)
CASH_RESERVE = mstr_data.get('cash', 0)

# Process convertible debt
convertible_debt = mstr_data.get('convertible_debt', [])
TOTAL_CONVERTIBLE_DEBT_PRINCIPAL = 0
CONVERTIBLE_DEBT_ISSUES = []

print("=" * 70)
print("CONVERTIBLE DEBT ANALYSIS")
print("=" * 70)

if convertible_debt:
    for i, debt in enumerate(convertible_debt, 1):
        # Extract principal - use 'notional' first (actual field name), then fallback to others
        principal = debt.get('notional', debt.get('principal', debt.get('face_value', debt.get('amount', 0))))
        # Extract conversion price - use 'strike_price' first (actual field name), then fallback to others
        conversion_price = debt.get('strike_price', debt.get('strikePrice', debt.get('strike', debt.get('conversion_price', debt.get('conversionPrice', debt.get('conversion_strike', 0))))))
        # Calculate conversion ratio from principal and conversion price
        conversion_ratio = 0
        if principal > 0 and conversion_price > 0:
            conversion_ratio = principal / conversion_price
        
        if principal > 0:
            TOTAL_CONVERTIBLE_DEBT_PRINCIPAL += principal
            
            # Get shares on conversion - use fetched data if available, otherwise calculate
            shares_on_conversion = debt.get('shares_on_conversion', 0)
            if shares_on_conversion == 0 or shares_on_conversion is None:
                # Calculate shares if converted (fallback if not fetched from shares page)
                if conversion_price > 0:
                    shares_on_conversion = principal / conversion_price
                elif conversion_ratio > 0:
                    shares_on_conversion = principal * conversion_ratio
            
            CONVERTIBLE_DEBT_ISSUES.append({
                'principal': principal,
                'conversion_price': conversion_price,
                'conversion_ratio': conversion_ratio,
                'shares_on_conversion': shares_on_conversion,
                'will_convert': conversion_price > 0 and MSTR_PRICE > conversion_price
            })
            
            will_convert_str = "YES" if (conversion_price > 0 and MSTR_PRICE > conversion_price) else "NO"
            print(f"Issue {i}:")
            print(f"  Principal: ${principal:,.0f}")
            print(f"  Conversion Price: ${conversion_price:,.2f}")
            if conversion_price > 0:
                print(f"  Current Stock Price: ${MSTR_PRICE:,.2f}")
                print(f"  Will Convert: {will_convert_str}")
                if MSTR_PRICE > conversion_price:
                    print(f"  Shares if Converted: {shares_on_conversion:,.0f}")
            print()
    
    print(f"Total Convertible Debt Principal: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
else:
    print("No convertible debt data found in strategy.com")
    print("Note: You may need to manually add debt data if available")

print("=" * 70)


CONVERTIBLE DEBT ANALYSIS
Issue 1:
  Principal: $1,010,000,000
  Conversion Price: $183.19
  Current Stock Price: $178.03
  Will Convert: NO

Issue 2:
  Principal: $3,000,000,000
  Conversion Price: $672.40
  Current Stock Price: $178.03
  Will Convert: NO

Issue 3:
  Principal: $2,000,000,000
  Conversion Price: $433.43
  Current Stock Price: $178.03
  Will Convert: NO

Issue 4:
  Principal: $800,000,000
  Conversion Price: $149.77
  Current Stock Price: $178.03
  Will Convert: YES
  Shares if Converted: 5,342,000

Issue 5:
  Principal: $603,750,000
  Conversion Price: $232.72
  Current Stock Price: $178.03
  Will Convert: NO

Issue 6:
  Principal: $800,000,000
  Conversion Price: $204.33
  Current Stock Price: $178.03
  Will Convert: NO

Total Convertible Debt Principal: $8,213,750,000


In [5]:
# ============================================================================
# CALCULATE NAV - SCENARIO 1: WITHOUT DEBT CONVERSION
# ============================================================================

# Calculate Bitcoin value
BITCOIN_VALUE = BITCOIN_HOLDINGS * CURRENT_BITCOIN_PRICE

# Calculate preferred stock obligations (par value)
PREFERRED_PAR_VALUE = 100
preferred_series = ['strc', 'strd', 'stre', 'strk', 'strf']
TOTAL_PREFERRED_PAR_VALUE = 0

for series in preferred_series:
    shares_key = f'{series}_shares'
    if shares_key in mstr_data:
        shares = mstr_data[shares_key]
        TOTAL_PREFERRED_PAR_VALUE += shares * PREFERRED_PAR_VALUE

# NAV without conversion = Bitcoin Value + Cash - Preferred Par Value - Convertible Debt
NAV_NO_CONVERSION = BITCOIN_VALUE + CASH_RESERVE - TOTAL_PREFERRED_PAR_VALUE - TOTAL_CONVERTIBLE_DEBT_PRINCIPAL

# Calculate NAV per share (without conversion)
if MSTR_SHARES > 0:
    NAV_PER_SHARE_NO_CONVERSION = NAV_NO_CONVERSION / MSTR_SHARES
else:
    NAV_PER_SHARE_NO_CONVERSION = 0

print("=" * 70)
print("NAV CALCULATION - SCENARIO 1: WITHOUT DEBT CONVERSION")
print("=" * 70)
print(f"Bitcoin Holdings: {BITCOIN_HOLDINGS:,} BTC")
print(f"Bitcoin Price: ${CURRENT_BITCOIN_PRICE:,.2f}")
print(f"Bitcoin Value: ${BITCOIN_VALUE:,.0f}")
print(f"Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"Total Preferred Stock Par Value: ${TOTAL_PREFERRED_PAR_VALUE:,.0f}")
print(f"Total Convertible Debt Principal: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"\nNAV = Bitcoin Value + Cash - Preferred Par Value - Convertible Debt")
print(f"NAV = ${BITCOIN_VALUE:,.0f} + ${CASH_RESERVE:,.0f} - ${TOTAL_PREFERRED_PAR_VALUE:,.0f} - ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"NAV = ${NAV_NO_CONVERSION:,.0f}")
print(f"\nMSTR Shares Outstanding: {MSTR_SHARES:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print("=" * 70)


NAV CALCULATION - SCENARIO 1: WITHOUT DEBT CONVERSION
Bitcoin Holdings: 818,869 BTC
Bitcoin Price: $79,339.44
Bitcoin Value: $64,968,605,846
Cash Reserve: $2,250,000,000
Total Preferred Stock Par Value: $13,538,673,300
Total Convertible Debt Principal: $8,213,750,000

NAV = Bitcoin Value + Cash - Preferred Par Value - Convertible Debt
NAV = $64,968,605,846 + $2,250,000,000 - $13,538,673,300 - $8,213,750,000
NAV = $45,466,182,546

MSTR Shares Outstanding: 355,898,000
NAV per Share: $127.75


In [6]:
# ============================================================================
# CALCULATE NAV - SCENARIO 2: WITH DEBT CONVERSION (ONLY IN-THE-MONEY)
# ============================================================================

# Calculate which debt will convert (only if conversion price < current stock price)
TOTAL_SHARES_FROM_CONVERSION = 0
TOTAL_DEBT_CONVERTED = 0

for debt in CONVERTIBLE_DEBT_ISSUES:
    if debt['will_convert']:
        TOTAL_SHARES_FROM_CONVERSION += debt['shares_on_conversion']
        TOTAL_DEBT_CONVERTED += debt['principal']

# New share count after conversion
MSTR_SHARES_AFTER_CONVERSION = MSTR_SHARES + TOTAL_SHARES_FROM_CONVERSION

# NAV with conversion = Bitcoin Value + Cash - Preferred Par Value - Non-converted Debt
# Converted debt becomes equity, so we don't subtract it
NAV_WITH_CONVERSION = BITCOIN_VALUE + CASH_RESERVE - TOTAL_PREFERRED_PAR_VALUE - (TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED)

# Calculate NAV per share (with conversion)
if MSTR_SHARES_AFTER_CONVERSION > 0:
    NAV_PER_SHARE_WITH_CONVERSION = NAV_WITH_CONVERSION / MSTR_SHARES_AFTER_CONVERSION
else:
    NAV_PER_SHARE_WITH_CONVERSION = 0

print("=" * 70)
print("NAV CALCULATION - SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print("=" * 70)
print(f"Current MSTR Shares: {MSTR_SHARES:,}")
print(f"Shares from Conversion: {TOTAL_SHARES_FROM_CONVERSION:,.0f}")
print(f"Total Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,.0f}")
print(f"\nDebt Converted: ${TOTAL_DEBT_CONVERTED:,.0f}")
print(f"Debt Remaining: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"\nBitcoin Value: ${BITCOIN_VALUE:,.0f}")
print(f"Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"Total Preferred Stock Par Value: ${TOTAL_PREFERRED_PAR_VALUE:,.0f}")
print(f"Remaining Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"\nNAV = Bitcoin Value + Cash - Preferred Par Value - Remaining Debt")
print(f"NAV = ${BITCOIN_VALUE:,.0f} + ${CASH_RESERVE:,.0f} - ${TOTAL_PREFERRED_PAR_VALUE:,.0f} - ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"NAV = ${NAV_WITH_CONVERSION:,.0f}")
print(f"\nMSTR Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print("=" * 70)


NAV CALCULATION - SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
Current MSTR Shares: 355,898,000
Shares from Conversion: 5,342,000
Total Shares After Conversion: 361,240,000

Debt Converted: $800,000,000
Debt Remaining: $7,413,750,000

Bitcoin Value: $64,968,605,846
Cash Reserve: $2,250,000,000
Total Preferred Stock Par Value: $13,538,673,300
Remaining Convertible Debt: $7,413,750,000

NAV = Bitcoin Value + Cash - Preferred Par Value - Remaining Debt
NAV = $64,968,605,846 + $2,250,000,000 - $13,538,673,300 - $7,413,750,000
NAV = $46,266,182,546

MSTR Shares After Conversion: 361,240,000
NAV per Share: $128.08


In [7]:
# ============================================================================
# CALCULATE STOCK PRICE AND PREMIUM (BOTH SCENARIOS)
# ============================================================================

STOCK_PRICE = MSTR_PRICE

# Premium calculations for both scenarios
if NAV_PER_SHARE_NO_CONVERSION > 0:
    PREMIUM_NO_CONVERSION = ((STOCK_PRICE - NAV_PER_SHARE_NO_CONVERSION) / NAV_PER_SHARE_NO_CONVERSION) * 100
else:
    PREMIUM_NO_CONVERSION = 0

if NAV_PER_SHARE_WITH_CONVERSION > 0:
    PREMIUM_WITH_CONVERSION = ((STOCK_PRICE - NAV_PER_SHARE_WITH_CONVERSION) / NAV_PER_SHARE_WITH_CONVERSION) * 100
else:
    PREMIUM_WITH_CONVERSION = 0

print("=" * 70)
print("STOCK PRICE AND PREMIUM CALCULATION")
print("=" * 70)
print(f"MSTR Stock Price: ${STOCK_PRICE:,.2f}")
print(f"\nSCENARIO 1: WITHOUT DEBT CONVERSION")
print(f"  NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print(f"  Premium: {PREMIUM_NO_CONVERSION:+.2f}%")
print(f"\nSCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print(f"  NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print(f"  Premium: {PREMIUM_WITH_CONVERSION:+.2f}%")
print("=" * 70)


STOCK PRICE AND PREMIUM CALCULATION
MSTR Stock Price: $178.03

SCENARIO 1: WITHOUT DEBT CONVERSION
  NAV per Share: $127.75
  Premium: +39.36%

SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
  NAV per Share: $128.08
  Premium: +39.00%


In [8]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("MSTR VALUATION SUMMARY")
print("=" * 70)

print(f"\nEnterprise Value (EV): ${ENTERPRISE_VALUE:,.0f}")
print(f"  - MSTR Common Stock Market Cap: ${MSTR_MARKET_CAP:,.0f} ({MSTR_SHARES:,} shares × ${MSTR_PRICE:,.2f})")
print(f"  - Preferred Stock Market Cap: ${TOTAL_PREFERRED_MARKET_CAP:,.0f}")
print(f"  - Total Equity Market Cap: ${EQUITY_MARKET_CAP:,.0f}")
print(f"  - Total Convertible Debt: ${TOTAL_DEBT:,.0f}")
print(f"  - Cash Reserve: ${CASH:,.0f}")
print(f"  - EV = Equity Market Cap + Debt - Cash = ${EQUITY_MARKET_CAP:,.0f} + ${TOTAL_DEBT:,.0f} - ${CASH:,.0f}")

print(f"\n" + "=" * 70)
print("SCENARIO 1: WITHOUT DEBT CONVERSION")
print("=" * 70)
print(f"Net Asset Value (NAV): ${NAV_NO_CONVERSION:,.0f}")
print(f"  - Bitcoin Value: ${BITCOIN_VALUE:,.0f} ({BITCOIN_HOLDINGS:,} BTC @ ${CURRENT_BITCOIN_PRICE:,.2f})")
print(f"  - Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"  - Preferred Par Value: ${TOTAL_PREFERRED_PAR_VALUE:,.0f}")
print(f"  - Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL:,.0f}")
print(f"\nMSTR Shares Outstanding: {MSTR_SHARES:,}")
print(f"NAV per Share: ${NAV_PER_SHARE_NO_CONVERSION:,.2f}")
print(f"Stock Price: ${STOCK_PRICE:,.2f}")
print(f"Premium: {PREMIUM_NO_CONVERSION:+.2f}%")

print(f"\n" + "=" * 70)
print("SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)")
print("=" * 70)
print(f"Net Asset Value (NAV): ${NAV_WITH_CONVERSION:,.0f}")
print(f"  - Bitcoin Value: ${BITCOIN_VALUE:,.0f} ({BITCOIN_HOLDINGS:,} BTC @ ${CURRENT_BITCOIN_PRICE:,.2f})")
print(f"  - Cash Reserve: ${CASH_RESERVE:,.0f}")
print(f"  - Preferred Par Value: ${TOTAL_PREFERRED_PAR_VALUE:,.0f}")
print(f"  - Remaining Convertible Debt: ${TOTAL_CONVERTIBLE_DEBT_PRINCIPAL - TOTAL_DEBT_CONVERTED:,.0f}")
print(f"  - Debt Converted to Equity: ${TOTAL_DEBT_CONVERTED:,.0f} ({TOTAL_SHARES_FROM_CONVERSION:,.0f} shares)")
print(f"\nMSTR Shares After Conversion: {MSTR_SHARES_AFTER_CONVERSION:,.0f}")
print(f"NAV per Share: ${NAV_PER_SHARE_WITH_CONVERSION:,.2f}")
print(f"Stock Price: ${STOCK_PRICE:,.2f}")
print(f"Premium: {PREMIUM_WITH_CONVERSION:+.2f}%")

print("\n" + "=" * 70)



MSTR VALUATION SUMMARY

Enterprise Value (EV): $82,377,749,550
  - MSTR Common Stock Market Cap: $63,360,520,506 (355,898,000 shares × $178.03)
  - Preferred Stock Market Cap: $13,053,479,045
  - Total Equity Market Cap: $76,413,999,550
  - Total Convertible Debt: $8,213,750,000
  - Cash Reserve: $2,250,000,000
  - EV = Equity Market Cap + Debt - Cash = $76,413,999,550 + $8,213,750,000 - $2,250,000,000

SCENARIO 1: WITHOUT DEBT CONVERSION
Net Asset Value (NAV): $45,466,182,546
  - Bitcoin Value: $64,968,605,846 (818,869 BTC @ $79,339.44)
  - Cash Reserve: $2,250,000,000
  - Preferred Par Value: $13,538,673,300
  - Convertible Debt: $8,213,750,000

MSTR Shares Outstanding: 355,898,000
NAV per Share: $127.75
Stock Price: $178.03
Premium: +39.36%

SCENARIO 2: WITH DEBT CONVERSION (IN-THE-MONEY ONLY)
Net Asset Value (NAV): $46,266,182,546
  - Bitcoin Value: $64,968,605,846 (818,869 BTC @ $79,339.44)
  - Cash Reserve: $2,250,000,000
  - Preferred Par Value: $13,538,673,300
  - Remaining Co